[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/monacofj/misda/blob/issue-75-optimization-benchmark/benchmarks/optimization.ipynb)

# MISDA — optimization benchmark

This notebook is a short proof-of-concept for the end-to-end optimization protocol: replace the full objective set by the subset selected by MISDA and ask whether comparable original-space optimization quality is reached in fewer generations.

The comparison is paired:

- **Full search** optimizes all original objectives and produces decision vectors `X_F`.
- **Reduced search** optimizes only the objectives selected by MISDA and produces decision vectors `X_R`.
- Both use NSGA-III with the same decision domain, initial decision population, population size, generation budget, and run seed.
- **Full** is the non-dominated original-space front `ND(F(X_F))`.
- **Full_r** is the non-dominated original-space front `ND(F(X_R))`: Reduced-search decision vectors re-evaluated on the original M-objective problem.
- This original-space re-evaluation is benchmark instrumentation only; it never feeds back into the Reduced search.

Before either search starts, the notebook reports MISDA's support, reconstruction, and observed Pareto preservation on the independent Sobol screening sample. These are discovery diagnostics, not a guarantee of global Pareto preservation.

MoeaBench's analytical calibration of the original MOP supplies the common Pareto ground truth and clinical baselines for optimization-quality measurement. Hypervolume is reported relative to that same calibrated ground-truth front.


In [ ]:
from pathlib import Path
import subprocess
import sys

# Test local code in a checkout. During PR review, Colab installs this PR branch; restore main before merge.
repo_root = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").exists()), None)
REMOTE_REF = "issue-75-optimization-benchmark"
target = f"{repo_root}[benchmarks]" if repo_root is not None else f"misda[benchmarks] @ git+https://github.com/monacofj/misda.git@{REMOTE_REF}"
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "--upgrade", target])


In [ ]:
import json
from pathlib import Path
import tempfile

import numpy as np
import pandas as pd
from IPython.display import display
from scipy.stats import qmc

import misda
import moeabench as mb
from moeabench.core.run import Population

# Short proof-of-concept configuration.
M = 10
SCREEN_POWER = 9          # 2**9 = 512 Sobol points
MISDA_SAMPLE = 2 ** SCREEN_POWER
POPULATION = 60
GENERATIONS = 50
BUDGET_CHECKPOINTS = (50, 100, 200, 400)
MISDA_SEED = 123
MOEA_SEED = 321
REF_DIRS_SEED = 456
HV_MC_SAMPLES = 20_000


## Experimental protocol

MISDA receives a reproducible **joint Sobol sample** of the original decision domain:

`X_screen → F(X_screen) → MISDA → ranked MIS candidate S`.

All decision variables vary jointly across the screening design. We deliberately do not use one-factor-at-a-time sampling here: this pilot aims to characterize the global objective structure, including interactions, rather than the local response around one fixed operating point. The screening stage is independent of both NSGA-III runs and does not use the analytical Pareto front. Before either optimizer starts, all discovered MISs receive observed dominance-preservation evaluation on **this same `Y_screen`**, and `dominance_preservation` selects the most conservative candidate. The selected MIS then receives explicit linear-reconstruction and observed Pareto-preservation evaluation. The full `ranking.report()` and selected MIS graph are displayed together with `NO_REDUNDANCY`, `SUPPORTED_REDUCTION`, or `UNSUPPORTED_REDUCTION`. An unsupported candidate is still carried forward so the benchmark can measure how wrong it becomes.

For optimization-quality measurement, the original MOP is calibrated through MoeaBench's default analytical `mop.calibrate()` protocol. The resulting sidecar is the canonical source for both the Pareto ground-truth reference (`gt_reference`) and the clinical baselines used by `clinic.audit`.

The optimization pair receives one explicitly generated initial decision population `X0`. The same matrix is passed to both NSGA-III instances through pymoo's `sampling=` interface. Reference-direction generation remains separate and dimension-appropriate; only the starting decision population is identical.

For each MOP we then run:

`Full search: X_F → F(X_F)`

`Reduced search: X_R → F_S(X_R)`

The comparison itself is always made back in the original objective space:

`Full = ND(F(X_F))`

`Full_r = ND(F(X_R))`

MISDA changes only the objective set seen by the Reduced search. The decision domain is identical in both searches. If some decision variables become inactive after objective reduction, that is reported as a consequence of the selected subset; those variables are **not removed from the MOEA**.

The Reduced MOP below is only an objective projection adapter: it delegates all mathematical evaluation to the original MoeaBench MOP and slices the resulting objective matrix. No DTLZ or DPF formula is duplicated.


### Budget calibration and statistical comparison

For DTLZ5, budget calibration comes **before** any Reduced search. One long Full NSGA-III trajectory is evaluated with MoeaBench's native `GD+`, `IGD+`, and relative `HV` trajectories at generations 50, 100, 200, and 400. These three metrics are sufficient here: GD+ measures proximity of the obtained front to the reference, IGD+ measures coverage of the reference front, and HV combines convergence and coverage.

`mb.stats` is intentionally deferred to the later multi-seed phase. These checkpoints are serially dependent observations from the same trajectory and must not be treated as independent samples for Mann–Whitney, KS, or A12 comparisons.


In [ ]:
class ObjectiveProjectionMOP(mb.mops.BaseMop):
    """Expose a subset of an existing MoeaBench MOP without duplicating it."""

    def __init__(self, source_mop, objective_indices):
        self.source_mop = source_mop
        self.objective_indices = tuple(int(i) for i in objective_indices)
        if len(self.objective_indices) < 2:
            raise ValueError("NSGA-III requires at least two selected objectives.")
        super().__init__(
            name=f"{source_mop.name}[MISDA]",
            M=len(self.objective_indices),
            N=source_mop.N,
            xl=np.asarray(source_mop.xl, dtype=float),
            xu=np.asarray(source_mop.xu, dtype=float),
        )

    def evaluation(self, X, n_ieq_constr=0):
        result = dict(self.source_mop.evaluation(X, n_ieq_constr))
        result["F"] = np.asarray(result["F"], dtype=float)[:, self.objective_indices]
        return result

    def ps(self, n_points=100):
        # Decision-space truth is unchanged; only the exposed objectives differ.
        return self.source_mop.ps(n_points)


In [ ]:
def _nd_front(F):
    """Return the non-dominated subset using MoeaBench's population algebra."""
    return np.asarray(Population(np.asarray(F, dtype=float)).non_dominated().objectives)


def _screen_misda(mop, *, power=SCREEN_POWER, seed=MISDA_SEED):
    """Joint low-discrepancy screening of the complete decision domain."""
    sampler = qmc.Sobol(d=mop.N, scramble=True, seed=seed)
    X_unit = sampler.random_base2(m=power)
    X = qmc.scale(
        X_unit,
        np.asarray(mop.xl, dtype=float),
        np.asarray(mop.xu, dtype=float),
    )
    F = np.asarray(mop.evaluation(X)["F"], dtype=float)
    frame = pd.DataFrame(F, columns=[f"f{i + 1}" for i in range(mop.M)])
    mis_set = misda.discover(frame, name=f"{mop.name} screening", seed=seed)

    # Conservative Y-only reduction choice: every MIS remains available, while
    # the selected view minimizes new observed dominance relations.
    mis_set.evaluate(metrics=("dominance",), candidates="all")
    ranking = misda.rank(
        mis_set,
        policy=misda.DOMINANCE_PRESERVATION,
    )
    selected = ranking.mis()
    return {
        "X": X,
        "F": frame,
        "mis_set": mis_set,
        "ranking": ranking,
        "assessment": ranking.assessment,
        "selected": selected,
        "indices": tuple(int(i) for i in selected.indices),
    }


def _diagnose_screening(screening, *, show_plots=True):
    """Evaluate the selected MIS on Y_screen, before any optimization or GT calibration."""
    mis_set = screening["mis_set"]
    ranking = screening["ranking"]
    selected = screening["selected"]
    mis_set.evaluate(metrics=("linear", "pareto"), candidates=selected)

    selected_support = mis_set.support_for(selected)
    assessment = ranking.assessment
    linear = selected.linear
    pareto = selected.pareto
    if linear is None or pareto is None:
        raise RuntimeError("Selected MIS is missing requested screening diagnostics.")

    lost = pareto.full_front_size - pareto.intersection_size
    n_screen = len(screening["F"])
    diagnostics = {
        "n_screen": n_screen,
        "original_dimension": mis_set.analysis.original_dimension,
        "latent_dimension": mis_set.analysis.latent_dimension,
        "structural_dimension": mis_set.analysis.structural_dimension,
        "selected_dimension": ranking.selected_dimension,
        "selected_indices": tuple(selected.indices),
        "reduction_status": assessment.status,
        "reduction_trustworthy": assessment.trustworthy,
        "new_dominance_rate": selected.dominance.new_dominance_rate,
        "new_dominance_pairs": selected.dominance.new_dominance_pairs,
        "group_support": mis_set.support.status if mis_set.support is not None else None,
        "selected_support": selected_support.status,
        "support_reasons": tuple(selected_support.reasons),
        "transitivity_excess": (
            selected_support.transitivity_excess
        ),
        "spectral_excess": (
            selected_support.spectral_excess
        ),
        "mean_r2": linear.mean_r2,
        "worst_r2": linear.worst_r2,
        "pareto_retention": pareto.retention,
        "pareto_validity": pareto.validity,
        "pareto_jaccard": pareto.jaccard,
        "front_loss": lost / pareto.full_front_size if pareto.full_front_size else None,
        "population_impact": lost / n_screen,
        "full_screen_nd": pareto.full_front_size,
        "reduced_screen_nd": pareto.reduced_front_size,
        "lost_screen_nd": lost,
    }

    print("\n=== Discovery evidence: MISDA on independent Sobol screening ===")
    print(
        f"Screening observations: {n_screen}; dominance evaluated MISs: "
        f"{len(mis_set)}/{len(mis_set)}; linear/Pareto evaluated MISs: 1/{len(mis_set)}"
    )
    print(ranking.report())
    print(
        f"Selected MIS support: {diagnostics['selected_support'] or 'N/A'}; "
        f"reasons: {', '.join(diagnostics['support_reasons']) or 'none'}"
    )
    print(
        f"Reduction assessment: {diagnostics['reduction_status']}; "
        f"trustworthy={diagnostics['reduction_trustworthy']}; "
        f"new-dominance rate={diagnostics['new_dominance_rate']:.6f}"
    )
    print(
        f"Screening front loss: {lost}/{pareto.full_front_size}; "
        f"population impact: {lost}/{n_screen}"
    )
    print("These metrics use Y_screen only; neither Pareto GT nor NSGA-III is consulted.")
    if show_plots:
        selected.graph_plot()
    return diagnostics



def _dtlz2_objective_irredundancy_check(mop):
    """Exact witness that every DTLZ2 objective is indispensable for dominance."""
    if mop.__class__.__name__ != "DTLZ2":
        raise ValueError("This analytical check is specific to DTLZ2.")
    axes = np.eye(mop.M, dtype=float)  # exact positive-orthant unit-sphere endpoints
    rows = []
    for omitted in range(mop.M):
        retained = tuple(i for i in range(mop.M) if i != omitted)
        witness = (omitted + 1) % mop.M
        full_pair = axes[[omitted, witness]]
        projected_pair = full_pair[:, retained]
        full_nd = len(_nd_front(full_pair))
        projected_nd = len(_nd_front(projected_pair))
        rows.append({
            "Omitted": f"f{omitted + 1}",
            "Witness retained": f"f{witness + 1}",
            "Full-space ND": full_nd,
            "Projected ND": projected_nd,
            "Dominance changed": full_nd == 2 and projected_nd == 1,
        })
    table = pd.DataFrame(rows)
    if not table["Dominance changed"].all():
        raise AssertionError("DTLZ2 analytical non-redundancy witness failed.")
    print("\n=== DTLZ2 analytical objective-reduction truth ===")
    print(
        "Every objective has an exact Pareto-front witness: removing it turns "
        "two originally incomparable axis points into a dominance relation."
    )
    display(table)
    return table


def _dtlz5_safe_reduction_check(
    mop,
    *,
    n_front=2000,
    stress_power=12,
    seed=MISDA_SEED,
):
    """Check the exact DTLZ5 safe pair f_(M-1), f_M plus an unsafe counterexample."""
    if mop.__class__.__name__ != "DTLZ5" or mop.M < 3:
        raise ValueError("This analytical check requires DTLZ5 with M >= 3.")

    safe_pair = (mop.M - 2, mop.M - 1)
    unsafe_pair = (0, mop.M - 1)

    # The analytical PF itself must remain fully nondominated under the safe pair.
    gt = np.asarray(mop.pf(n_points=n_front), dtype=float)
    full_gt_nd = len(_nd_front(gt))
    safe_gt_nd = len(_nd_front(gt[:, safe_pair]))

    # Stress the stronger statement needed for optimization: every off-front
    # point must be weakly dominated, in the reduced pair, by its g=0 counterpart
    # with the same x1. DTLZ5's trailing g variables begin at index M-1.
    sampler = qmc.Sobol(d=mop.N, scramble=True, seed=seed)
    X = qmc.scale(
        sampler.random_base2(m=stress_power),
        np.asarray(mop.xl, dtype=float),
        np.asarray(mop.xu, dtype=float),
    )
    F = np.asarray(mop.evaluation(X)["F"], dtype=float)
    X_front = X.copy()
    X_front[:, mop.M - 1:] = 0.5
    F_front = np.asarray(mop.evaluation(X_front)["F"], dtype=float)

    reduced = F[:, safe_pair]
    reduced_front = F_front[:, safe_pair]
    weak = np.all(reduced_front <= reduced + 1e-12, axis=1)
    strict = np.any(reduced_front < reduced - 1e-12, axis=1)
    matched_front_dominates = weak & strict

    # Degeneracy alone does not make an arbitrary pair safe. For M=10,
    # choosing x1=0, all intermediate position variables=1 and all g variables=1
    # gives an off-front point with f_M=0 and f1 below the true-front endpoint.
    X_bad = np.zeros((1, mop.N), dtype=float)
    X_bad[:, 1:mop.M - 1] = 1.0
    X_bad[:, mop.M - 1:] = 1.0
    X_bad_front = X_bad.copy()
    X_bad_front[:, mop.M - 1:] = 0.5
    F_bad = np.asarray(mop.evaluation(X_bad)["F"], dtype=float)
    F_bad_front = np.asarray(mop.evaluation(X_bad_front)["F"], dtype=float)
    bad = F_bad[:, unsafe_pair][0]
    bad_front = F_bad_front[:, unsafe_pair][0]
    unsafe_witness = bool(
        np.all(bad <= bad_front + 1e-12)
        and np.any(bad < bad_front - 1e-12)
    )

    table = pd.DataFrame([
        {
            "Reduction": f"f{safe_pair[0] + 1}, f{safe_pair[1] + 1}",
            "Analytical PF ND": full_gt_nd,
            "Projected PF ND": safe_gt_nd,
            "Matched off-front checks": len(X),
            "All matched g=0 points dominate": bool(matched_front_dominates.all()),
            "Role": "optimization-safe pair",
        },
        {
            "Reduction": f"f{unsafe_pair[0] + 1}, f{unsafe_pair[1] + 1}",
            "Analytical PF ND": full_gt_nd,
            "Projected PF ND": len(_nd_front(gt[:, unsafe_pair])),
            "Matched off-front checks": 1,
            "All matched g=0 points dominate": False,
            "Role": "unsafe counterexample exists",
        },
    ])
    if not (
        full_gt_nd == n_front
        and safe_gt_nd == n_front
        and matched_front_dominates.all()
        and unsafe_witness
    ):
        raise AssertionError("DTLZ5 analytical reduction checks failed.")

    print("\n=== DTLZ5 analytical objective-reduction truth ===")
    display(table)
    print(
        f"Unsafe witness for f1, f{mop.M}: "
        f"off-front={bad.tolist()}, front counterpart={bad_front.tolist()}"
    )
    return {
        "safe_pair": safe_pair,
        "unsafe_pair": unsafe_pair,
        "gt": gt,
        "table": table,
        "matched_front_dominates": matched_front_dominates,
        "unsafe_witness": unsafe_witness,
        "unsafe_values": {"off_front": bad, "front": bad_front},
    }


def _dpf1_gt_projection_check(mop, *, n_points=2000, observed_pair=(1, 7)):
    """Test two objective pairs on an independent analytical DPF1 Pareto-set sample.

    This preflight does not run an optimizer, calibrate FAIR, or affect MISDA selection.
    The sampled GT test complements, but does not replace, analytical reasoning.
    """
    if mop.__class__.__name__ != "DPF1" or mop.D != 2:
        raise ValueError("Analytical pair check requires DPF1 with D=2.")
    if n_points < 2:
        raise ValueError("At least two analytical GT points are required.")
    X_ps = np.asarray(mop.ps(n_points=n_points), dtype=float)
    gt = np.asarray(mop.evaluation(X_ps)["F"], dtype=float)
    if gt.shape != (n_points, mop.M):
        raise ValueError("Unexpected shape for DPF1 analytical GT.")
    full_nd_count = len(_nd_front(gt))
    rows = []
    for label, indices in (
        ("Analytical base", (0, 1)),
        ("Previously selected MISDA", tuple(int(i) for i in observed_pair)),
    ):
        if len(indices) != 2 or min(indices) < 0 or max(indices) >= mop.M:
            raise ValueError(f"Invalid DPF1 objective pair {indices}.")
        projected_count = len(_nd_front(gt[:, indices]))
        rows.append({
            "Pair": label,
            "Objectives": ", ".join(f"f{i + 1}" for i in indices),
            "Full GT ND": full_nd_count,
            "Projected GT ND": projected_count,
            "All GT points retained": projected_count == full_nd_count == n_points,
        })
    table = pd.DataFrame(rows)
    print("\n=== DPF1: analytical Pareto-front projection preflight ===")
    print(f"Analytical GT: {n_points} original-MOP evaluations (no optimizer).")
    display(table)
    print("Finite-front evidence only; arbitrary off-front dominance is not certified.")
    return {"X_ps": X_ps, "gt": gt, "table": table}


def _active_decision_dimension(mop, selected_indices):
    """Analytical support count for the two pilot MOPs; never changes the MOEA domain."""
    if mop.__class__.__name__ == "DTLZ2":
        active = set(range(mop.M - 1, mop.N))  # g variables affect every objective
        for i in selected_indices:
            if i == 0:
                active.update(range(mop.M - 1))
            else:
                active.update(range(mop.M - i))
        return len(active)

    if mop.__class__.__name__ == "DPF1":
        # Both base objectives depend on x1 and the shared g term; every projected
        # objective is a linear combination of those base objectives.
        return mop.N

    return None


def _calibrated_ground_truth(mop):
    """Calibrate the original MOP and return its canonical MoeaBench GT."""
    sidecar = (
        Path(tempfile.gettempdir())
        / f"misda-{mop.name}-M{mop.M}-calibration.json"
    )
    # Default calibration generates the analytical GT plus FAIR baselines
    # across MoeaBench's supported K grid. K is inferred later from each
    # original-space front being audited.
    mop.calibrate(
        source_baseline=str(sidecar),
        force=True,
    )
    payload = json.loads(sidecar.read_text(encoding="utf-8"))
    gt = np.asarray(payload["gt_reference"], dtype=float)
    if gt.ndim != 2 or gt.shape[1] != mop.M:
        raise ValueError(
            f"Calibrated GT for {mop.name} has shape {gt.shape}; expected (*, {mop.M})."
        )
    return gt, str(sidecar)


def _paired_initial_population(mop, *, size=POPULATION, seed=MOEA_SEED):
    """One explicit decision population shared by Full and Reduced."""
    rng = np.random.default_rng(seed)
    return rng.uniform(
        np.asarray(mop.xl, dtype=float),
        np.asarray(mop.xu, dtype=float),
        size=(size, mop.N),
    )


def _canonical_rows(X):
    """Order-independent representation for paired-population assertions."""
    X = np.asarray(X, dtype=float)
    order = np.lexsort(X.T[::-1])
    return X[order]


def _original_space_history(exp, original_mop):
    """Re-evaluate every generation's decision vectors on the original MOP."""
    return [
        _nd_front(original_mop.evaluation(np.asarray(X, dtype=float))["F"])
        for X in exp[0].history("x")
    ]


def _history_evaluations(exp):
    """Population evaluations represented by the recorded generational history."""
    return int(sum(np.asarray(X).shape[0] for X in exp[0].history("x")))


def _metric_value(metric, front, gt):
    return float(metric(np.asarray(front), ref=np.asarray(gt), progress=False))


def _igdplus_history(fronts, gt, label):
    values = [
        _metric_value(mb.metrics.igdplus, front, gt)
        for front in fronts
    ]
    return mb.metrics.MetricMatrix(
        np.asarray(values, dtype=float)[:, None],
        metric_name="IGD+",
        source_name=label,
    )


def _budget_checkpoint_table(metric_map, checkpoints):
    """Read one long run at named 1-based generation checkpoints."""
    return pd.DataFrame([
        {
            "Generation": int(generation),
            **{
                label: float(matrix.mean(int(generation) - 1))
                for label, matrix in metric_map.items()
            },
        }
        for generation in checkpoints
    ])


def _full_budget_calibration(name, mop, *, checkpoints=BUDGET_CHECKPOINTS):
    """Run Full once to the largest checkpoint and inspect native MoeaBench histories."""
    checkpoints = tuple(sorted(set(int(value) for value in checkpoints)))
    if not checkpoints or checkpoints[0] < 1:
        raise ValueError("checkpoints must contain positive generation numbers.")

    X0 = _paired_initial_population(mop)
    full = mb.experiment(
        mop=mop,
        moea=mb.moeas.NSGA3(
            population=POPULATION,
            generations=checkpoints[-1],
            seed=MOEA_SEED,
            ref_dirs_seed=REF_DIRS_SEED,
            sampling=X0.copy(),
        ),
    )
    full.name = f"{name} — Full budget calibration"
    full.run(repeat=1, silent=True)
    np.testing.assert_allclose(
        _canonical_rows(full[0].history("x")[0]),
        _canonical_rows(X0),
    )

    gt, calibration_sidecar = _calibrated_ground_truth(mop)
    metric_map = {
        "GD+": mb.metrics.gdplus(full, ref=gt, progress=False),
        "IGD+": mb.metrics.igdplus(full, ref=gt, progress=False),
        "HV (relative to GT)": mb.metrics.hypervolume(
            full,
            ref=gt,
            mode="auto",
            scale="rel",
            n_samples=HV_MC_SAMPLES,
            mc_seed=MOEA_SEED,
            progress=False,
        ),
    }
    checkpoint_table = _budget_checkpoint_table(metric_map, checkpoints)

    print(f"\n=== {name}: Full-only budget calibration ===")
    print(
        f"One Full trajectory to {checkpoints[-1]} generations; "
        f"checkpoints={checkpoints}; population={POPULATION}."
    )
    print(
        "Choose the budget from stabilization of MoeaBench GD+, IGD+, and "
        "relative HV. No Reduced search is run in this calibration."
    )
    display(checkpoint_table)
    for label, matrix in metric_map.items():
        mb.view.history(matrix, title=f"{name}: Full {label} over generations")

    return {
        "name": name,
        "mop": mop,
        "full": full,
        "gt": gt,
        "calibration_sidecar": calibration_sidecar,
        "metrics": metric_map,
        "checkpoints": checkpoint_table,
        "evaluations": _history_evaluations(full),
    }


def _final_metrics(full_front, full_r_front, gt):
    gd_full = _metric_value(mb.metrics.gdplus, full_front, gt)
    gd_full_r = _metric_value(mb.metrics.gdplus, full_r_front, gt)
    igd_full = _metric_value(mb.metrics.igdplus, full_front, gt)
    igd_full_r = _metric_value(mb.metrics.igdplus, full_r_front, gt)

    hv_kwargs = dict(
        ref=np.asarray(gt),
        mode="auto",
        scale="rel",
        n_samples=HV_MC_SAMPLES,
        mc_seed=MOEA_SEED,
        progress=False,
    )
    hv_full = float(mb.metrics.hypervolume(np.asarray(full_front), **hv_kwargs))
    hv_full_r = float(mb.metrics.hypervolume(np.asarray(full_r_front), **hv_kwargs))

    return pd.DataFrame(
        {
            "Full": [gd_full, igd_full, hv_full],
            "Full_r": [gd_full_r, igd_full_r, hv_full_r],
            "Full_r - Full": [
                gd_full_r - gd_full,
                igd_full_r - igd_full,
                hv_full_r - hv_full,
            ],
        },
        index=["GD+", "IGD+", "HV (relative to GT)"],
    )


In [ ]:
optimization_results = {}


def run_optimization_case(name, mop):
    screening = _screen_misda(mop)
    screening_diagnostics = _diagnose_screening(screening)
    selected_indices = screening["indices"]
    selected_labels = [f"f{i + 1}" for i in selected_indices]
    active_decision_dimension = _active_decision_dimension(mop, selected_indices)

    reduced_mop = ObjectiveProjectionMOP(mop, selected_indices)

    # Pairing is explicit: both formulations receive exactly the same X0.
    X0 = _paired_initial_population(mop)
    initial_original = np.asarray(mop.evaluation(X0)["F"], dtype=float)

    full = mb.experiment(
        mop=mop,
        moea=mb.moeas.NSGA3(
            population=POPULATION,
            generations=GENERATIONS,
            seed=MOEA_SEED,
            ref_dirs_seed=REF_DIRS_SEED,
            sampling=X0.copy(),
        ),
    )
    full.name = f"{name} — Full"

    reduced = mb.experiment(
        mop=reduced_mop,
        moea=mb.moeas.NSGA3(
            population=POPULATION,
            generations=GENERATIONS,
            seed=MOEA_SEED,
            ref_dirs_seed=REF_DIRS_SEED,
            sampling=X0.copy(),
        ),
    )
    reduced.name = f"{name} — Reduced"

    full.run(repeat=1, silent=True)
    reduced.run(repeat=1, silent=True)

    # NSGA-III may reorder the initial population during its objective-specific
    # survival step. Membership, not row order, is the pairing invariant.
    np.testing.assert_allclose(
        _canonical_rows(full[0].history("x")[0]),
        _canonical_rows(X0),
    )
    np.testing.assert_allclose(
        _canonical_rows(reduced[0].history("x")[0]),
        _canonical_rows(X0),
    )

    # Pareto ground truth and clinical baselines come from MoeaBench calibration
    # of the original M-objective problem, never from the Reduced formulation.
    gt, calibration_sidecar = _calibrated_ground_truth(mop)

    full_history = _original_space_history(full, mop)
    full_r_history = _original_space_history(reduced, mop)
    full_front = full_history[-1]
    full_r_front = full_r_history[-1]

    metrics = _final_metrics(full_front, full_r_front, gt)

    diag_full = mb.clinic.audit(
        full_front,
        ground_truth=gt,
        source_baseline=calibration_sidecar,
        initial_data=initial_original,
        problem=name,
    )
    diag_full.experiment_name = "Full"

    diag_full_r = mb.clinic.audit(
        full_r_front,
        ground_truth=gt,
        source_baseline=calibration_sidecar,
        initial_data=initial_original,
        problem=name,
    )
    diag_full_r.experiment_name = "Full_r"

    igd_full = _igdplus_history(full_history, gt, "Full")
    igd_full_r = _igdplus_history(full_r_history, gt, "Full_r")

    full_evaluations = _history_evaluations(full)
    reduced_evaluations = _history_evaluations(reduced)
    assert full_evaluations == reduced_evaluations

    result = {
        "name": name,
        "mop": mop,
        "screening": screening,
        "screening_diagnostics": screening_diagnostics,
        "selected_indices": selected_indices,
        "selected_labels": selected_labels,
        "active_decision_dimension": active_decision_dimension,
        "initial_X": X0,
        "initial_original": initial_original,
        "reduced_mop": reduced_mop,
        "full": full,
        "reduced": reduced,
        "gt": gt,
        "calibration_sidecar": calibration_sidecar,
        "full_history": full_history,
        "full_r_history": full_r_history,
        "full_front": full_front,
        "full_r_front": full_r_front,
        "metrics": metrics,
        "diag_full": diag_full,
        "diag_full_r": diag_full_r,
        "igd_full": igd_full,
        "igd_full_r": igd_full_r,
        "evaluations": full_evaluations,
    }
    optimization_results[name] = result

    print(f"\n=== Search consequence: {name}, original {mop.M}-objective space ===")
    print(
        f"{name}: ranked MIS candidate under {screening['ranking'].policy}: "
        f"{mop.M} → {len(selected_indices)} — "
        f"{screening_diagnostics['reduction_status']}"
    )
    print(
        f"Selected support: {screening_diagnostics['selected_support'] or 'N/A'}; "
        f"new-dominance rate={screening_diagnostics['new_dominance_rate']:.6f}"
    )
    print(f"Selected objectives: {', '.join(selected_labels)}")
    print(f"MoeaBench calibrated GT: {len(gt)} points in {gt.shape[1]} objectives")
    if active_decision_dimension is not None:
        print(
            f"Decision dimension: original={mop.N}, "
            f"active after objective reduction={active_decision_dimension} "
            "(measured only; MOEA domain unchanged)"
        )
    print(
        f"Budget per treatment: generations={len(full[0].history('x'))}, "
        f"evaluations={full_evaluations}, population={POPULATION}"
    )
    print(
        f"Original-space ND cardinality: Full={len(full_front)}, "
        f"Full_r={len(full_r_front)}"
    )
    print(
        f"FAIR baseline K (snapped): Full={diag_full.diagnostic_context['k']}, "
        f"Full_r={diag_full_r.diagnostic_context['k']}"
    )
    display(metrics)
    mb.view.topology(
        full_front,
        full_r_front,
        gt=gt,
        show_gt=True,
        objectives=[0, 1, 2],
        labels=["Full", "Full_r"],
        title=f"{name}: Full vs Full_r in original objective space (f1–f3 projection)",
    )

    mb.view.radar(
        diag_full,
        diag_full_r,
        title=f"{name}: Full vs Full_r clinical quality in original objective space",
    )

    mb.view.history(
        igd_full,
        igd_full_r,
        title=f"{name}: Full vs Full_r IGD+ convergence in original objective space",
    )

    return result


## Pilot battery

This first run is intentionally short. It is a proof of concept for the complete protocol, not a definitive performance study.

- **DTLZ2** — regular smooth **negative control for objective removal**. Its Pareto front is the positive-orthant unit hypersphere. Every individual objective is indispensable to full-space dominance, even though the front has geometric dimension `M-1`. MISDA-specific latent/structural truth is deliberately not fabricated from that geometry.
- **DTLZ5** — classical degenerate positive-control candidate. We establish a concrete optimization-safe pair, run MISDA independently on global Sobol screening, and calibrate the Full NSGA-III budget before any Reduced comparison.
- **DPF1** — explicit degenerate projection from an intrinsic base of `D=2` objectives to `M=10`. This construction dimension is useful external context, but it is not silently treated as a declared MISDA structural truth.

### Analytical properties (external to MISDA truth)

These MOPs have known mathematical structure, but **geometric dimension is not MISDA's latent or structural dimension**. These are external facts, not synthetic `latent_expected` or `structural_expected` declarations:

| MOP, M=10 | Global objective-image dimension | Geometric Pareto-front dimension |
|---|---:|---:|
| DTLZ2 | 10 | 9 |
| DPF1 (D=2) | 2 | 1 |

A separate **optimization-reduction truth** is now established where it can be proved exactly:

| MOP | Proven statement for objective removal |
|---|---|
| DTLZ2 | No non-empty proper subset of objectives preserves all Pareto-front dominance relations. |
| DTLZ5 | The pair `f9, f10` (for `M=10`) is optimization-safe; arbitrary pairs need not be. |
| DPF1 | The two analytical base objectives preserve the full construction; the previously observed `f2, f8` pair preserves the analytical front but not arbitrary off-front dominance. |

For DPF1, the base objectives are `u=(1+g)x1/2` and `v=(1+g)(1-x1)/2`; the remaining eight objectives are fixed positive-weight linear combinations of `u` and `v`. The original base pair therefore preserves dominance for the full MOP. On its analytical Pareto front, `g=0`, so `u+v=1/2`. An **arbitrary** pair of projected objectives, even if linearly reconstructive, is not thereby certified to preserve dominance. Similarly, DTLZ2's nine-dimensional Pareto-front manifold does not certify that any of its objectives can be omitted without changing dominance.

These properties describe the MOP, not the graph inferred from the particular `Y_screen`. Internal MISDA support does not guarantee that the selected subset preserves the quality of a newly optimized front.

A separate notion of ground truth is used for optimization quality: MoeaBench's analytical `mop.calibrate()` protocol is run on the original MOP, and its sidecar supplies the Pareto GT and clinical baselines. That calibrated Pareto ground truth is the common reference for GD+, IGD+, relative HV, and the clinical Q-scores. The comparison is Full vs Full_r in the original M-objective space; the Reduced search itself is never compared directly in its lower-dimensional objective space.

DTLZ5 is not yet subjected to a Full-vs-Reduced comparison. First we inspect its MISDA result and establish a defensible Full budget from one long run using native MoeaBench GD+, IGD+, and relative-HV trajectories. `mb.stats` is reserved for the later repeated-seed phase, where independent run distributions exist. DTLZ7, DPF3 and DPF5 remain future extensions.


In [ ]:
PROBLEMS = {
    "DTLZ2": mb.mops.DTLZ2(M=M),
    "DTLZ5": mb.mops.DTLZ5(M=M),
    "DPF1": mb.mops.DPF1(M=M, D=2, K=5),
}

PROBLEM_CONTEXT = {
    "DTLZ2": {
        "misda_truth": None,
        "external_context": f"Pareto manifold dimension = {M - 1}",
    },
    "DTLZ5": {
        "misda_truth": None,
        "external_context": "Degenerate Pareto-front dimension = 1; safe pair established separately",
    },
    "DPF1": {
        "misda_truth": None,
        "external_context": "Intrinsic base objective dimension D = 2",
    },
}

for name, mop in PROBLEMS.items():
    print(f"{name}: M={mop.M}, N={mop.N} — {PROBLEM_CONTEXT[name]['external_context']}")


## DTLZ2 — negative control for objective removal

DTLZ2 is non-degenerate in the sense relevant here: its Pareto front is the
positive-orthant unit hypersphere,

`sum_i f_i^2 = 1`, with `f_i >= 0`.

Its geometric dimension is `M-1`, but that does **not** make one objective
removable. For any omitted objective `f_j` and any retained objective `f_k`,
the axis points `e_j` and `e_k` are both valid Pareto-front points and are
incomparable in the full space. After dropping `f_j`, `e_j` projects to the
zero vector and dominates `e_k`. Therefore every non-empty proper objective
subset changes a Pareto-front dominance relation.

This is an exact external truth about objective removal, not a declaration
about MISDA latent or structural dimension. The preflight below records the
witness for every objective before MISDA/NSGA-III results are interpreted.


In [ ]:
dtlz2_objective_truth = _dtlz2_objective_irredundancy_check(PROBLEMS["DTLZ2"])


In [ ]:
dtlz2 = run_optimization_case("DTLZ2", PROBLEMS["DTLZ2"])


## DTLZ5 — analytical positive-control candidate

DTLZ5 has a one-dimensional Pareto-front image for `M>2`, but degeneracy alone
does not imply that arbitrary objectives can be dropped. Here we prove a
specific safe reduction for `M=10` before interpreting MISDA or adding a Reduced optimization comparison.

Let `t=1+g >= 1`. On the true front, `g=0`,
`theta_2=...=theta_9=pi/4`, and the final two objectives are

`f9 = cos(theta_1)/sqrt(2)`,  
`f10 = sin(theta_1)`.

Off the front,

`f9 = t cos(theta_1) sin(theta_2)`,  
`f10 = t sin(theta_1)`,

with `theta_2 >= pi/(4t)`. Because
`t sin(pi/(4t)) >= 1/sqrt(2)`, the point with the same `x1` and `g=0`
weakly improves both retained objectives, with strict improvement when
`g>0`. Thus `{f9,f10}` has the same Pareto set/front as the original DTLZ5
for the purpose of minimization.

The check also constructs an explicit counterexample for `{f1,f10}`:
an off-front point can improve `f1` while keeping `f10=0`, so not every
pair suggested by the one-dimensional front geometry is optimization-safe.


### MISDA screening, then Full-only budget calibration

The analytical result above is **not** supplied to MISDA. The same 512-point global Sobol screening used elsewhere is analyzed independently. All discovered MISs are ranked by observed dominance preservation; the selected MIS then receives reconstruction and same-sample Pareto diagnostics. Only afterwards do we compare that Y-only answer and its trust annotation with the external analytical truth.

Before any DTLZ5 Reduced run, the Full baseline is run once to 400 generations. Native MoeaBench GD+, IGD+ and relative-HV histories are inspected at generations 50, 100, 200 and 400 to choose a defensible budget.


In [ ]:
dtlz5_objective_truth = _dtlz5_safe_reduction_check(PROBLEMS["DTLZ5"])


In [ ]:
dtlz5_screening = _screen_misda(PROBLEMS["DTLZ5"])
dtlz5_screening_diagnostics = _diagnose_screening(dtlz5_screening)


### Full-only budget calibration

This is not a Full-vs-Reduced experiment. One long Full NSGA-III run to 400
generations is inspected at 50, 100, 200 and 400 with native MoeaBench GD+,
IGD+ and relative-HV histories. A Reduced search is deliberately postponed
until the Full baseline has a defensible budget.

`mb.stats` is reserved for independent repeated runs; checkpoints from one
trajectory are serially dependent.


In [ ]:
dtlz5_budget = _full_budget_calibration("DTLZ5", PROBLEMS["DTLZ5"])


## DPF1 — linear degenerate projection

DPF1 is a positive control for high-dimensional objectives generated from a low-dimensional base front.

### Analytical projection preflight

Before running DPF1, use MoeaBench's analytical Pareto-set sampler to evaluate 2,000
original-space points, then test whether the same points remain non-dominated
when projected onto the basic pair `f1, f2` or the observed MISDA pair `f2, f8`.
This independent check is not used by MISDA or either optimizer.

On the DPF1 analytical front, `f1+f2=1/2`. For these specific projected weights,
`f8=a*f1+b*f2`, with `a>b>0`; therefore, `f8` decreases as `f2` increases.
The pair `f2, f8` preserves trade-offs on the exact analytical front,
even though it can discard nondominated points sampled away from that front.


In [ ]:
dpf1_gt_projection = _dpf1_gt_projection_check(PROBLEMS["DPF1"])


In [ ]:
dpf1 = run_optimization_case("DPF1", PROBLEMS["DPF1"])


# Suite summary

The summary keeps the scientific axes separate: discovery evidence on Y_screen, final original-space quality, and convergence. It distinguishes aggregate first-tie-group support from the selected MIS's own support; missing reconstruction values mean there were no eliminated targets. Screening metrics cannot be interpreted as out-of-sample or global optimization guarantees. A lower GD+/IGD+ and a higher HV indicate better final approximation, but the table deliberately reports values and deltas rather than declaring a winner.



In [ ]:
summary_rows = []
for name, result in optimization_results.items():
    metrics = result["metrics"]
    discovery = result["screening_diagnostics"]
    summary_rows.append(
        {
            "Problem": name,
            "Objectives": result["mop"].M,
            "Selected": len(result["selected_indices"]),
            "MISDA latent": discovery["latent_dimension"],
            "MISDA structural": discovery["structural_dimension"],
            "Group support": discovery["group_support"],
            "Selected support": discovery["selected_support"],
            "Support reasons": ", ".join(discovery["support_reasons"]) or "none",
            "Screen mean R²": discovery["mean_r2"],
            "Screen worst R²": discovery["worst_r2"],
            "Screen Pareto retention": discovery["pareto_retention"],
            "Screen Pareto validity": discovery["pareto_validity"],
            "Screen Pareto Jaccard": discovery["pareto_jaccard"],
            "Screen front loss": discovery["front_loss"],
            "Screen population impact": discovery["population_impact"],
            "Decision dim.": result["mop"].N,
            "Active decision dim.": result["active_decision_dimension"],
            "Generations": len(result["full"][0].history("x")),
            "Evaluations": result["evaluations"],
            "|Full|": len(result["full_front"]),
            "|Full_r|": len(result["full_r_front"]),
            "FAIR K Full": result["diag_full"].diagnostic_context["k"],
            "FAIR K Full_r": result["diag_full_r"].diagnostic_context["k"],
            "GD+ Full": metrics.loc["GD+", "Full"],
            "GD+ Full_r": metrics.loc["GD+", "Full_r"],
            "IGD+ Full": metrics.loc["IGD+", "Full"],
            "IGD+ Full_r": metrics.loc["IGD+", "Full_r"],
            "HV Full": metrics.loc["HV (relative to GT)", "Full"],
            "HV Full_r": metrics.loc["HV (relative to GT)", "Full_r"],
            "Final IGD+ Δ": metrics.loc["IGD+", "Full_r - Full"],
        }
    )

optimization_summary = pd.DataFrame(summary_rows)
# Confront discovery diagnostics with original-space Full/Full_r outcomes.
optimization_confrontation = optimization_summary[
    [
        "Problem", "Objectives", "Selected", "MISDA latent", "MISDA structural",
        "Group support", "Selected support", "Support reasons",
        "Screen mean R²", "Screen worst R²", "Screen Pareto retention",
        "Screen Pareto validity", "Screen Pareto Jaccard",
        "Screen front loss", "Screen population impact",
        "|Full|", "|Full_r|", "IGD+ Full", "IGD+ Full_r",
        "HV Full", "HV Full_r",
    ]
]
display(optimization_confrontation)
